In [1]:
import urllib.request

# 네이버 영화리뷰 트레인/테스트 데이터 다운로드
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt", filename="ratings_train.txt")
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt", filename="ratings_test.txt")

print("다운로드 완료!")

다운로드 완료!


In [2]:
import pandas as pd

# 1. TSV(탭으로 구분된 파일) 형태의 데이터를 데이터프레임으로 불러오기
train_data = pd.read_csv('ratings_train.txt', sep='\t')
test_data = pd.read_csv('ratings_test.txt', sep='\t')

# 2. 데이터 크기 및 샘플 확인
print("Train Data 개수:", len(train_data))
print("Test Data 개수:", len(test_data))

# 상위 5개 데이터 확인
train_data.head()

Train Data 개수: 150000
Test Data 개수: 50000


,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [3]:
import numpy as np

train_data.drop_duplicates(subset=['document'], inplace=True) # document(리뷰 내용) 컬럼에서 똑같은 내용의 중복 리뷰 제거
train_data.dropna(how='any', inplace=True)                    # 내용이 없는(Null/NaN) 데이터 제거

test_data.drop_duplicates(subset=['document'], inplace=True)
test_data.dropna(how='any', inplace=True)

print(f"전처리 후 Train 데이터 개수: {len(train_data)}")
print(f"전처리 후 Test 데이터 개수: {len(test_data)}")

# 2. 정제된 리뷰만 nsmc_corpus.txt 파일로 추출
with open('nsmc_corpus.txt', 'w', encoding='utf-8') as f:
    for line in train_data['document']:
        f.write(line + '\n')

print("🎉 Step 1 완료: 'nsmc_corpus.txt' 말뭉치 생성 성공!")

전처리 후 Train 데이터 개수: 146182
전처리 후 Test 데이터 개수: 49157
🎉 Step 1 완료: 'nsmc_corpus.txt' 말뭉치 생성 성공!


리뷰 내용 중 같은 내용의 리뷰 제거  
및 전처리 후 데이터 개수 확인

In [4]:
import sentencepiece as spm

# 1. 학습 설정값 정의
corpus_file = 'nsmc_corpus.txt'  # Step 1에서 만든 말뭉치 파일
model_prefix = 'nsmc_spm'         # 생성될 모델 파일의 이름 prefix
vocab_size = 8000                 # 단어장에 저장할 서브워드 개수

# 2. SentencePiece 모델 학습 명령
spm.SentencePieceTrainer.Train(
    f"--input={corpus_file} "
    f"--model_prefix={model_prefix} "
    f"--vocab_size={vocab_size} "
    f"--model_type=unigram "                   # 모델 타입 unigram
    f"--pad_id=0 --pad_piece=[PAD] "           # 문장 길이를 일정하게 맞추기 위한 빈칸용 토큰
    f"--unk_id=1 --unk_piece=[UNK] "           # 단어장에 없는 모르는 단어 대체용 토큰
    f"--bos_id=2 --bos_piece=[BOS] "           # 문장의 시작을 알리는 토큰
    f"--eos_id=3 --eos_piece=[EOS]"            # 문장의 끝을 알리는 토큰
)

print("🎉 Step 2 완료: 'nsmc_spm.model' 및 'nsmc_spm.vocab' 파일이 생성되었습니다!")

🎉 Step 2 완료: 'nsmc_spm.model' 및 'nsmc_spm.vocab' 파일이 생성되었습니다!


--pad_id=0  --pad_piece=[PAD]   # 0번 인덱스에 [PAD] 토큰 할당  
--unk_id=1  --unk_piece=[UNK]   # 1번 인덱스에 [UNK] 토큰 할당  
--bos_id=2  --bos_piece=[BOS]   # 2번 인덱스에 [BOS] 토큰 할당  
--eos_id=3  --eos_piece=[EOS]   # 3번 인덱스에 [EOS] 토큰 할당

### 📌 [참고] SentencePiece 특수 토큰(Special Tokens) 역할 및 설정 이유

SentencePiece 모델 학습 과정에서 지정한 특수 토큰들은 자연어 처리(NLP) 모델이 텍스트 시퀀스를 올바르게 인식하고 처리하기 위한 필수 표준 사전 정의 값입니다.

---

#### 1. 주요 특수 토큰 종류 및 역할

| 토크나이저 설정 | 토큰 명칭 | ID (인덱스) | 역할 및 세부 설명 |
| :--- | :---: | :---: | :--- |
| `--pad_piece=[PAD]` | **PAD** (Padding) | `0` | **문장 길이 균일화**: 서로 다른 길이의 리뷰 문장들을 배치(Batch) 단위로 모아 학습할 때, 빈 공간을 채워 길이를 동일하게 맞추는 토큰입니다. |
| `--unk_piece=[UNK]` | **UNK** (Unknown) | `1` | **미학습 단어 대체**: 단어장(Vocab, Size=8,000)에 등록되지 않은 오탈자, 희귀 단어, 신조어가 입력될 경우 이를 안전하게 대체하는 토큰입니다. |
| `--bos_piece=[BOS]` | **BOS** (Beginning of Sequence) | `2` | **문장 시작 알림**: 모델에게 텍스트 시퀀스의 시작 지점을 알려주는 토큰입니다. (주로 Seq2Seq 번역 및 생성 모델에서 필수 활용) |
| `--eos_piece=[EOS]` | **EOS** (End of Sequence) | `3` | **문장 종결 알림**: 텍스트 시퀀스가 끝났음을 알려주는 토큰으로, 모델이 문장 생성을 멈출 시점을 파악하게 합니다. |

---

#### 2. 고정 인덱스(0~3) 설정 이유

1. **딥러닝 프레임워크(Keras/PyTorch) 연동 최적화**
   * Keras의 `Embedding` 레이어나 `pad_sequences` 함수는 기본적으로 **`0`번 인덱스를 패딩(PAD) 값으로 가정**하고 연산을 수행합니다.
   * 이에 맞춰 `--pad_id=0`으로 지정함으로써 추가적인 인덱스 맵핑 오버헤드 없이 안전하게 모델을 구성할 수 있습니다.

2. **Out-of-Vocabulary(OOV) 예외 처리**
   * 자연어 데이터의 특성상 단어장에 없는 새로운 문자열이 들어왔을 때 프로그램이 에러(Crash)를 내지 않고 **`1`번(`[UNK]`)으로 변환하여 안정적으로 처리**하도록 보장합니다.

3. **표준 NLP 파이프라인 규격 준수**
   * SentencePiece 학습 시 인덱스 규격(`0: PAD`, `1: UNK`, `2: BOS`, `3: EOS`)을 고정해두면, 추후 사전 학습된 다른 토크나이저나 언어 모델(Transformer 계열 등)과의 호환성을 손쉽게 유지할 수 있습니다.

In [5]:
import numpy as np

# 1. 학습된 SentencePiece 모델 로드
s = spm.SentencePieceProcessor()
s.Load('nsmc_spm.model')

# 2. sp_tokenize 함수 정의
def sp_tokenize(s, corpus, max_len=50):
    tensor = []

    for sentence in corpus:
        encoded = s.EncodeAsIds(sentence)

        # Truncation (자르기)
        if len(encoded) > max_len:
            encoded = encoded[:max_len]
        # Padding (채우기)
        else:
            encoded = encoded + [0] * (max_len - len(encoded))

        tensor.append(encoded)

    return np.array(tensor)

# 3. 전체 데이터셋 변환 수행
X_train = sp_tokenize(s, train_data['document'], max_len=50)
y_train = train_data['label'].values

X_test = sp_tokenize(s, test_data['document'], max_len=50)
y_test = test_data['label'].values

print("🎉 Step 3 완료!")
print("X_train 행렬 형태(Shape):", X_train.shape)
print("X_test 행렬 형태(Shape):", X_test.shape)

🎉 Step 3 완료!
X_train 행렬 형태(Shape): (146182, 50)
X_test 행렬 형태(Shape): (49157, 50)


### 💡 텍스트 전처리 및 벡터화 파이프라인 (Tokenization vs Embedding)

자연어 처리(NLP) 파이프라인은 **'토크나이저의 정수 인덱싱'**과 **'모델 내 Embedding 레이어의 차원 벡터화'** 2단계로 구분되어 작동합니다.

---

#### 1. 토크나이저 단계 (`sp_tokenize` 함수)
* **역할**: 원본 텍스트를 단어장(Vocab) 기준의 **정수 ID(Integer Index) 시퀀스**로 맵핑하고 길이를 정규화합니다.
* **작동 과정**:
  1. **인코딩**: 텍스트 문장을 미리 학습된 단어장 번호로 변환 (예: `"영화 재밌다"` $\rightarrow$ `[42, 108, 9]`)
  2. **Truncation**: `max_len(50)` 초과 분량 자르기
  3. **Padding**: 부족한 길이는 `0`(`[PAD]` 토큰)으로 채워 고정 길이(50) 형성
* **특징**: 메모리 효율성을 위해 텍스트를 고차원 벡터로 직접 바꾸지 않고, 가벼운 1차원 정수 배열 형태(`shape=(50,)`)로 출력합니다.

---

#### 2. 모델 벡터화 단계 (Keras `Embedding` Layer)
* **역할**: 정수 ID 시퀀스를 받아서 의미적 거리가 반영된 **밀집 벡터(Dense Vector)**로 변환합니다.
* **작동 과정**:
  * 모델의 첫 번째 레이어인 `Embedding(vocab_size=8000, embedding_dim=128)`이 실행됩니다.
  * 입력된 정수 번호(예: `42`)를 128차원의 실수 공간에 맵핑하여 `(50, 128)` 차원의 2차원 행렬을 생성합니다.
* **특징**: 단어의 의미적 유착 관계(벡터 표현)는 토크나이저가 아닌 **GPU 기반의 딥러닝 모델 학습(Backpropagation) 과정에서 최적화**됩니다.

---

#### 🔄 전체 데이터 흐름 요약

```text
[입력 텍스트]
  │
  ▼  (1단계: sp_tokenize)
[정수 ID 배열] : [42, 108, 9, 0, 0, ..., 0]  (Shape: 50,)
  │
  ▼  (2단계: Embedding Layer)
[밀집 벡터 행렬] : [[0.15, -0.42, ..., 0.88], ...] (Shape: 50 × 128)
  │
  ▼  (3단계: LSTM & Classification)
[감정 예측] : 긍정(1) 또는 부정(0) 확률 출력

In [6]:
import tensorflow as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# 1. Hyperparameters 설정
vocab_size = 8000    # Step 2에서 설정한 단어장 크기
embedding_dim = 128  # 단어를 표현할 밀집 벡터의 차원 수
hidden_units = 128   # LSTM 레이어의 은닉 상태(Hidden State) 크기

# 2. LSTM 모델 구조 설계
model = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=50),
    LSTM(hidden_units),
    Dropout(0.5),                  # 오버피팅(과적합) 방지
    Dense(1, activation='sigmoid') # 긍정(1)/부정(0) 이진 분류
])

# 3. 모델 컴파일 - 어떻게 학습을 진행하고 시험을 볼 것인가?
model.compile(
    optimizer='adam',              # 가장 성능이 좋고 널리 쓰이는 표준 최적화 알고리즘
    loss='binary_crossentropy',    # 오차(Loss)를 계산
    metrics=['accuracy']           # 정확도 점수 표기 방식
)

# 4. 콜백 함수 설정 (조기 종료 및 최선 모델 저장)
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=4)
mc = ModelCheckpoint('best_spm_model.h5', monitor='val_accuracy', mode='max', verbose=1, save_best_only=True)

# 5. 모델 학습 진행
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.2, # Train 데이터의 20%를 검증용으로 활용
    callbacks=[es, mc]
)

# 저장된 최선의 모델을 불러오거나 현재 모델로 평가
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"\n🎉 최종 Test Accuracy: {test_acc * 100:.2f}%")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/15
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.4995 - loss: 0.6935
Epoch 1: val_accuracy improved from None to 0.63293, saving model to best_spm_model.h5



Epoch 1: finished saving model to best_spm_model.h5
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 23s 9ms/step - accuracy: 0.5143 - loss: 0.6908 - val_accuracy: 0.6329 - val_loss: 0.6609
Epoch 2/15
1821/1828 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6903 - loss: 0.5848
Epoch 2: val_accuracy improved from 0.63293 to 0.84588, saving model to best_spm_model.h5



Epoch 2: finished saving model to best_spm_model.h5
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.7630 - loss: 0.4823 - val_accuracy: 0.8459 - val_loss: 0.3472
Epoch 3/15
1821/1828 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8662 - loss: 0.3144
Epoch 3: val_accuracy improved from 0.84588 to 0.85953, saving model to best_spm_model.h5



Epoch 3: finished saving model to best_spm_model.h5
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.8654 - loss: 0.3137 - val_accuracy: 0.8595 - val_loss: 0.3252
Epoch 4/15
1823/1828 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8905 - loss: 0.2607
Epoch 4: val_accuracy did not improve from 0.85953
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.8867 - loss: 0.2678 - val_accuracy: 0.8594 - val_loss: 0.3336
Epoch 5/15
1825/1828 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9088 - loss: 0.2201
Epoch 5: val_accuracy did not improve from 0.85953
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.9029 - loss: 0.2316 - val_accuracy: 0.8544 - val_loss: 0.3570
Epoch 6/15
1822/1828 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.9238 - loss: 0.1869
Epoch 6: val_accuracy did not improve from 0.85953
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - accuracy: 0.9193 - loss: 0.1970 - val_accuracy: 0.8553 - val_loss: 0.3759
Epoch 7/15
1825/1828 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/s

```
# 4. 콜백 함수 설정 (조기 종료 및 최선 모델 저장)
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=4)
mc = ModelCheckpoint('best_spm_model.h5', monitor='val_accuracy', mode='max', verbose=1, save_best_only=True)
```
monitor='val_accuracy'
mode='max'
save_best_only=True
검증 정확도가 가장 높은 순간을 저장

verbose=1
화면에 진행 상태 바 를 보여주며 loss , accuracy, val_loss , val_accuracy 를 보여줌

loss = 훈련 오차  
accuracy = 훈련 정확도  
val_loss = 모의고사 오차  
val_accuracy = 모의고사 정확도  

warnning이 보이긴 하는데 학습에는 문제 없습니다.
만약 변경한다면 이렇게 변경시켜줍니다.
```
# 수정 전
Embedding(vocab_size, embedding_dim, input_length=50)

# 수정 후
Embedding(vocab_size, embedding_dim)

# 수정 전

mc = ModelCheckpoint('best_spm_model.h5', monitor='val_accuracy', mode='max', verbose=1, save_best_only=True)

# 수정 후 (.h5 대신 .keras 권장 확장자 사용)

mc = ModelCheckpoint('best_spm_model.keras', monitor='val_accuracy', mode='max', verbose=1, save_best_only=True)
```

In [11]:
from keras.models import load_model
# 저장해둔 최고 성능 모델(Epoch 3) 불러오기
best_model = load_model('best_spm_model.h5')

# 최고 모델로 테스트 데이터 재평가
test_loss, test_acc = best_model.evaluate(X_test, y_test)
print(f"\n최고 모델 Test Loss: {test_loss:.4f}")
print(f"최고 모델 Test Accuracy: {test_acc * 100:.2f}%")

1537/1537 ━━━━━━━━━━━━━━━━━━━━ 8s 5ms/step - accuracy: 0.8544 - loss: 0.3319

최고 모델 Test Loss: 0.3319
최고 모델 Test Accuracy: 85.44%


In [12]:
import sentencepiece as spm
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.models import load_model

# ==========================================
# 1. SentencePiece (BPE 방식) 토크나이저 학습
# ==========================================
temp_file = 'nsmc_corpus.txt' # 기존 학습에 사용했던 전처리된 텍스트 파일명

spm.SentencePieceTrainer.train(
    f"--input={temp_file} "
    f"--model_prefix=nsmc_bpe "
    f"--vocab_size=8000 "
    f"--model_type=bpe "          # Unigram 대신 BPE 지정
    f"--pad_id=0 --unk_id=1 --bos_id=2 --eos_id=3"
)

# BPE 토크나이저 로드
sp_bpe = spm.SentencePieceProcessor()
sp_bpe.load('nsmc_bpe.model')

# ==========================================
# 2. 데이터 토큰화 및 패딩
# ==========================================
def sp_bpe_tokenize(sp, corpus, max_len=50):
    tensor = []
    for line in corpus:
        tensor.append(sp.encode_as_ids(str(line)))
    return pad_sequences(tensor, maxlen=max_len, padding='post')

# X_train_raw, X_test_raw는 기존 전처리 완료된 텍스트 리스트
X_train_bpe = sp_bpe_tokenize(sp_bpe, train_data['document'])
X_test_bpe = sp_bpe_tokenize(sp_bpe, test_data['document'])

# ==========================================
# 3. 동일한 LSTM 모델 구축 및 학습
# ==========================================
vocab_size = 8000
embedding_dim = 128
hidden_units = 128

model_bpe = Sequential([
    Embedding(vocab_size, embedding_dim),
    LSTM(hidden_units),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model_bpe.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

es_bpe = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=4)
mc_bpe = ModelCheckpoint('best_bpe_model.h5', monitor='val_accuracy', mode='max', verbose=1, save_best_only=True)

print("--- BPE 기반 모델 학습 시작 ---")
history_bpe = model_bpe.fit(
    X_train_bpe, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    callbacks=[es_bpe, mc_bpe]
)

# ==========================================
# 4. 저장된 최적 BPE 모델 평가
# ==========================================
best_bpe_model = load_model('best_bpe_model.h5')
bpe_test_loss, bpe_test_acc = best_bpe_model.evaluate(X_test_bpe, y_test)

print(f"\n🎉 [BPE 최적 모델] Test Loss: {bpe_test_loss:.4f}")
print(f"🎉 [BPE 최적 모델] Test Accuracy: {bpe_test_acc * 100:.2f}%")

--- BPE 기반 모델 학습 시작 ---
Epoch 1/15
1823/1828 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5048 - loss: 0.6932
Epoch 1: val_accuracy improved from None to 0.60276, saving model to best_bpe_model.h5



Epoch 1: finished saving model to best_bpe_model.h5
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 16s 8ms/step - accuracy: 0.5144 - loss: 0.6901 - val_accuracy: 0.6028 - val_loss: 0.6541
Epoch 2/15
1824/1828 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6196 - loss: 0.6239
Epoch 2: val_accuracy improved from 0.60276 to 0.84773, saving model to best_bpe_model.h5



Epoch 2: finished saving model to best_bpe_model.h5
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.7101 - loss: 0.5249 - val_accuracy: 0.8477 - val_loss: 0.3527
Epoch 3/15
1827/1828 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8638 - loss: 0.3225
Epoch 3: val_accuracy improved from 0.84773 to 0.85840, saving model to best_bpe_model.h5



Epoch 3: finished saving model to best_bpe_model.h5
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.8635 - loss: 0.3210 - val_accuracy: 0.8584 - val_loss: 0.3259
Epoch 4/15
1822/1828 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8880 - loss: 0.2697
Epoch 4: val_accuracy improved from 0.85840 to 0.86192, saving model to best_bpe_model.h5



Epoch 4: finished saving model to best_bpe_model.h5
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.8851 - loss: 0.2753 - val_accuracy: 0.8619 - val_loss: 0.3281
Epoch 5/15
1826/1828 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9047 - loss: 0.2305
Epoch 5: val_accuracy did not improve from 0.86192
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9013 - loss: 0.2375 - val_accuracy: 0.8573 - val_loss: 0.3319
Epoch 6/15
1823/1828 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9235 - loss: 0.1898
Epoch 6: val_accuracy did not improve from 0.86192
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9186 - loss: 0.2011 - val_accuracy: 0.8536 - val_loss: 0.3627
Epoch 7/15
1827/1828 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9400 - loss: 0.1558
Epoch 7: val_accuracy did not improve from 0.86192
1828/1828 ━━━━━━━━━━━━━━━━━━━━ 16s 9ms/step - accuracy: 0.9354 - loss: 0.1654 - val_accuracy: 0.8476 - val_loss: 0.4227
Epoch 7: early stopping


1537/1537 ━━━━━━━━━━━━━━━━━━━━ 7s 4ms/step - accuracy: 0.8599 - loss: 0.3332

🎉 [BPE 최적 모델] Test Loss: 0.3332
🎉 [BPE 최적 모델] Test Accuracy: 85.99%


## 📊 [루브릭 3] SentencePiece 서브워드 분할 방식(Unigram vs BPE) 비교 분석

### 1. 모델 Performance 비교

| 토크나이저 방식 (`model_type`) | Vocab Size | Epochs (Best) | Test Loss | **Test Accuracy** |
| :--- | :---: | :---: | :---: | :---: |
| **Unigram** | 8,000 | 3 | `0.3319` | **`85.44%`** |
| **BPE** (Byte Pair Encoding) | 8,000 | 4 | `0.3332` | **`85.99%`** |

---

처음에 실수로 cpu로 돌려서 시간이 35분 걸렸습니다.  
코랩에서 돌릴 땐 gpu로 바꿔서 돌려야 한다는 걸 잊지 말아야 겠습니다.

계속 돌리다 보니까 가장 정확도가 높았던 Epochs 로 테스트를 돌리고 싶어서 코드를 추가하여 진행했습니다.
